This notebook was run using GPU T4 from google collab doing a direct conexion VSCode -> Google Colab, therefore, any local file was uploaded to drive storage

In [58]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

from pyarrow import parquet as pq

from time import time
import json
import random
import math

In [59]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
class LoadLanguageData(IterableDataset):

    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        worker_info = get_worker_info()

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            if worker_info is None: # Main process (num_workers = 0)
                iter_start = 0
                iter_end = len(rows)
            else:
                per_worker = int(math.ceil(len(rows) / float(worker_info.num_workers))) # Number of rows to be processed by each worker
                worker_id = worker_info.id
                iter_start = worker_id * per_worker
                iter_end = min(iter_start + per_worker, len(rows))

            for i in range(iter_start, iter_end):

                row = rows[i]

                yield {
                    'input_ids' : torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[2]], dtype=torch.long)
                }


def wrapper_collate_batch(pad_value_in, pad_value_out):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]

        length_input  = torch.tensor([len(x) for x in input_list], dtype=torch.long)
        length_output = torch.tensor([len(x) for x in output_list],dtype=torch.long)

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value_in)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value_out)

        return (length_input, input_padded, length_output, output_padded)

    return collate_batch


def load_tokenizer(path:str) -> tuple:

    with open(path) as f:
        tokenizer = json.load(f)
    
    index2word = {int(key): value for key, value in tokenizer["inverse_vocab"].items()}
    special_tokens = tokenizer["special_tokens"]

    return special_tokens, index2word


def index2word(input_tensor: torch.Tensor, vocab_mapper: dict, unk_token:int) -> list:

    indices_matrix = input_tensor.cpu().numpy()

    output = [
        [vocab_mapper.get(idx, vocab_mapper[unk_token]) for idx in sentence]
        for sentence in indices_matrix
    ]
    
    return output

In [67]:
sp_tokens_eng, vocab_idx2word_eng = load_tokenizer(path + "/data/main/vocabs/eng_tokenizer.json")
sp_tokens_spa, vocab_idx2word_spa = load_tokenizer(path + "/data/main/vocabs/spa_tokenizer.json")

pad_value_eng = sp_tokens_eng["[PAD]"]
pad_value_spa = sp_tokens_spa["[PAD]"]

In [68]:
train_path = path + "/data/main/spa-eng/train_spa_eng"
train_schema = pq.ParquetDataset(train_path)
train_cols = train_schema.schema.names

test_path = path + "/data/main/spa-eng/test_spa_eng"
test_schema = pq.ParquetDataset(test_path)
test_cols = test_schema.schema.names

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = False if device.type == "cpu" else True

train_data = LoadLanguageData(train_path, train_cols)
train_loader = DataLoader(train_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa), num_workers=2, pin_memory=pin_memory)

test_data = LoadLanguageData(test_path, test_cols)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa), num_workers=2, pin_memory=pin_memory)

In [ ]:
import numpy as np

for eng_len, eng_in, spa_len, spa_out in train_loader:
    break

print(eng_len[:3])
print(np.array(index2word(eng_in, vocab_idx2word_eng, sp_tokens_eng["[UNK]"]))[:3], end="\n\n\n")
print(spa_len[:3])
print(np.array(index2word(spa_out,vocab_idx2word_spa, sp_tokens_spa["[UNK]"]))[:3])

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size:int, embedding_size:int, hidden_size:int, layers:int, p_dropout:float) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)

        self.gru = nn.GRU(embedding_size, hidden_size, layers, bidirectional=True, batch_first=True)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        x = self.embedding(x)
        x = self.dropout(x)

        # output will have shape
        # (batch, seq_length, hidden_size * 2 if bidireccional else hidden_size)
        # hidden will have shape 
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        outputs, hidden = self.gru(x)

        # (batch, 1, hidden_size + hidden_size)
        x = torch.concat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

        # Expected hidden dimension (e.g if it'll be used for another GRU)
        x = torch.unsqueeze(x, dim=0)

        # output for cross-attention
        # x as hidden state

        return outputs, x


class DotProductAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    # Cross-Attention
    def forward(self, hidden:torch.Tensor, encoder_outputs:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:

        # Q -> hidde_state del Decoder (primera iteración hidden del encoder)
        # K y V -> output del Encoder
        Q = hidden.transpose(0, 1) # (batch, 1, encoder_outputs.size(-1))
        K = encoder_outputs
        K_t = K.transpose(1, 2) # (batch, encoder_outputs.size(-1), seq_len)

        d_k = Q.size(-1)
        energy_scaled = torch.bmm(Q, K_t) / (d_k ** 0.5)

        att_weights = torch.softmax(energy_scaled, dim=2)
        context = torch.bmm(att_weights, K).squeeze(1)

        return att_weights, context.squeeze(1)


class BridgeEncoderDecoder(nn.Module):
    def __init__(self, encoder_hidden_size:int, decoder_hidden_size:int):
        super().__init__()

        # Proyection to match hidden_state and output between encoder-decoder
        self.bridge_hidden  = nn.Linear(encoder_hidden_size, decoder_hidden_size)
        self.output_proyect = nn.Linear(encoder_hidden_size, decoder_hidden_size)
    
    def forward(self, encoder_hidden_state, encoder_outputs):

        hidden_proyection = torch.tanh(self.bridge_hidden(encoder_hidden_state))
        output_proyection = self.output_proyect(encoder_outputs)

        return hidden_proyection, output_proyection


class Decoder(nn.Module):
    def __init__(self, vocab_size:int, embedding_size:int, hidden_size:int, p_dropout:float) -> None:
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)
    
        self.gru = nn.GRU(embedding_size + hidden_size, hidden_size, batch_first=True)
        # GRU [output] will be concatenated with [context] and [embedding] for more information
        # It's a Skip Connection like Deep Output Layer or Output Fusion.
        self.ol = nn.Linear(hidden_size + hidden_size + embedding_size, vocab_size)

    def forward(self, x:torch.Tensor, hidden_state:torch.Tensor, context:torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        x = x.unsqueeze(1) # (batch, 1)

        embedded = self.embedding(x)
        embedded = self.dropout(embedded) # (batch, 1, embedding_size)

        context = context.unsqueeze(1) # (batch, 1, hidden_size)

        gru_input = torch.concatenate((embedded, context), dim=2) # (batch, 1, embedding_size + hidden_size)
        output, hidden = self.gru(gru_input, hidden_state)

        prediction = torch.concat((output, context, embedded), dim=2)
        prediction = self.ol(prediction).squeeze(1)

        return output, hidden, prediction


class Seq2Seq(nn.Module):
    def __init__(self, encoder:Encoder, bridge:BridgeEncoderDecoder, attention: DotProductAttention, decoder:Decoder, output_vocab_size:int, teacher_forcing_ratio:float = 0.5, train_mode:bool = True) -> None:
        super().__init__()

        self.train_mode = train_mode
        self.output_vocab_size = output_vocab_size
        self.teacher_forcing_ratio = teacher_forcing_ratio

        self.encoder = encoder
        self.bridge = bridge
        self.att = attention
        self.decoder = decoder

    def forward(self, x_1, x_2):

        self.train(self.train_mode) if self.train_mode else self.eval()

        enc_output, enc_hidden = self.encoder(x_1)

        enc_hidden_pro, enc_output_pro = self.bridge(enc_hidden, enc_output)

        _, context = self.att(enc_hidden_pro, enc_output_pro)

        batch_size = x_2.size(-2)
        target_len = x_2.size(-1)

        logits = torch.zeros(target_len, batch_size, self.output_vocab_size).to(enc_hidden.device)

        input_batch_token_step = x_2[:, 0]
        hidden_dec = enc_hidden_pro

        for idx_token in range(1, target_len):

            _, hidden_dec, pred = self.decoder(input_batch_token_step, hidden_dec, context) # In the first iteration we use encoder hidden_state
            _, context = self.att(hidden_dec, enc_output_pro)

            # Predictions (probabilities) to optimize
            logits[idx_token] = pred

            # Tokens to the next iteration (either true tokens or predicted tokens)
            teacher_force = random.random() < self.teacher_forcing_ratio
            top = pred.argmax(1)
            input_batch_token_step = x_2[:, idx_token] if teacher_force else top

        return logits

In [9]:
# Input_english -> Encoder -> hidden, output
# hidden, output -> Bridge -> hidden_p, output_p
# hidden_p, output_p -> Atention -> context
# context, hidden_p, Input_spanish -> 

# for ...
#   -> Decoder -> hidden_dec, output_dec, pred 

In [10]:
######################
### Getting Batch ####
######################
eng_batch, spa_o_batch = next(iter(train_loader))


#######################
### Testing Encoder ###
#######################
encoder = Encoder(
    vocab_size=len(vocab_idx2word_eng), 
    embedding_size=300,
    hidden_size=50,
    layers=1,
    p_dropout=0.3
)
outputs_e, hidden_e = encoder(eng_batch)

print(f"Batch: {eng_batch.shape}")
print(f"Output_enc: {outputs_e.shape}")
print(f"Hidden_enc: {hidden_e.shape}")

Batch: torch.Size([64, 11])
Output_enc: torch.Size([64, 11, 100])
Hidden_enc: torch.Size([1, 64, 100])


In [11]:
############################################
### Testing Proyection (encoder results) ###
############################################
bridge_conexion = BridgeEncoderDecoder(
    encoder_hidden_size=hidden_e.size(-1),
    decoder_hidden_size=200
)
hidden_pro, outputs_pro = bridge_conexion(hidden_e, outputs_e)

print("")
print(outputs_pro.shape)
print(hidden_pro.shape)


torch.Size([64, 11, 200])
torch.Size([1, 64, 200])


In [12]:
###########################################
### Testing Attention (encoder results) ###
###########################################
dot_prod_att = DotProductAttention()
att, context = dot_prod_att(hidden_pro, outputs_pro)

print("")
print(f"Att Weights: {att.shape}")
print(f"Context: {context.shape}")


Att Weights: torch.Size([64, 1, 11])
Context: torch.Size([64, 200])


In [13]:
#####################################
### Testing Decoder + Predictions ###
#####################################
decoder = Decoder(
    vocab_size = len(vocab_idx2word_spa), 
    embedding_size = 300,
    hidden_size = 200,
    p_dropout = 0.3
)

input_batch_token_step = spa_o_batch[:, 0]
hidden_dec = hidden_pro

for idx_token in range(1, spa_o_batch.size(-1)):
    output_dec, hidden_dec, pred = decoder(input_batch_token_step, hidden_dec, context) # In the first iteration we use encoder hidden_state
    _, context = dot_prod_att(hidden_dec, outputs_pro)

    input_batch_token_step = spa_o_batch[:, idx_token]

print(spa_o_batch.shape)
print(input_batch_token_step.unsqueeze(1).shape) # Internamente se hace unsqueeze en decoder
print(output_dec.shape)
print(hidden_dec.shape)
print(context.shape)

torch.Size([64, 13])
torch.Size([64, 1])
torch.Size([64, 1, 200])
torch.Size([1, 64, 200])
torch.Size([64, 200])


In [14]:
############################################
### Testing Decoder + Predictions + Loss ###
############################################

batch_size = spa_o_batch.size(-2)
target_len = spa_o_batch.size(-1)
vocab_size = len(vocab_idx2word_spa)

outputs = torch.zeros(target_len, batch_size, vocab_size)

input_batch_token_step = spa_o_batch[:, 0]
hidden_dec = hidden_pro

teacher_forcing_ratio = 0.5

for idx_token in range(1, target_len):

    output_dec, hidden_dec, pred = decoder(input_batch_token_step, hidden_dec, context) # In the first iteration we use encoder hidden_state
    _, context = dot_prod_att(hidden_dec, outputs_pro)

    # Predictions (probabilities) to optimize
    outputs[idx_token] = pred

    # Tokens to the next iteration (either true tokens or predicted tokens)
    teacher_force = random.random() < teacher_forcing_ratio
    top = pred.argmax(1)
    input_batch_token_step = spa_o_batch[:, idx_token] if teacher_force else top


criterion = nn.CrossEntropyLoss(ignore_index=sp_tokens_spa["[PAD]"])

# Removing first token. It was not predicted
logits = outputs[1:].view(-1, vocab_size)
targets= spa_o_batch[:, 1:].contiguous().view(-1)

loss = criterion(logits, targets)

In [15]:
##########################
### Testing Full Model ###
##########################

hidden_size_enc = 256 
hidden_size_dec = 512

encoder = Encoder(
    vocab_size=len(vocab_idx2word_eng), 
    embedding_size=300,
    hidden_size=hidden_size_enc,
    layers=1,
    p_dropout=0.3
)
decoder = Decoder(
    vocab_size = len(vocab_idx2word_spa), 
    embedding_size = 300,
    hidden_size = hidden_size_dec,
    p_dropout = 0.3
)
bridge  = BridgeEncoderDecoder(
    encoder_hidden_size = hidden_size_enc * 2,
    decoder_hidden_size = hidden_size_dec
)
attention = DotProductAttention()

machine_translation_model = Seq2Seq(
    encoder=encoder,
    bridge=bridge,
    attention=attention,
    decoder=decoder,
    output_vocab_size=len(vocab_idx2word_spa),
    teacher_forcing_ratio=0.5,
    train_mode = True
)
machine_translation_model.to(device)
machine_translation_model.train()

#####################
### Running model ###
#####################

epoch = 10
alpha = 0.0001

criterion = nn.CrossEntropyLoss(ignore_index=sp_tokens_spa["[PAD]"])
optimizer = torch.optim.Adam(machine_translation_model.parameters(), lr=alpha)

loss_registry = []
time_cumulative = 0

for ep in range(epoch):

    current_time = time()

    for batch_number, (eng_batch, spa_batch) in enumerate(train_loader, 1):

        eng_batch = eng_batch.to(device)
        spa_batch = spa_batch.to(device)

        optimizer.zero_grad()

        logits = machine_translation_model(eng_batch, spa_batch)

        # Removing first token. It was not predicted
        logits = logits[1:].contiguous().view(-1, vocab_size)
        targets= spa_batch[:, 1:].contiguous().view(-1)

        loss = criterion(logits, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(machine_translation_model.parameters(), max_norm=1.0)
        optimizer.step()

        if (batch_number % 25) == 0:
            print(f"Epoch/Batch: {ep + 1}/{batch_number}", end="\r", flush=True)

            ppl = torch.exp(loss)
            loss_registry.append(loss.item())
            time_cumulative += time() - current_time

            print(f"Epoch/Batch: {ep + 1}/{batch_number} - Loss: {loss.item():.4f} - PPL: {ppl:.4f} - Time: {time_cumulative:.4f}")

print("END")

KeyboardInterrupt: 

In [49]:
embedded = nn.Embedding(len(vocab_idx2word_eng), 300)(eng_batch)

length = (eng_batch != sp_tokens_eng["[PAD]"]).sum(dim=1)
input_packed = pack_padded_sequence(embedded, length, batch_first=True, enforce_sorted=False)

packed_output, hidden = nn.GRU(embedded.size(-1), 256)(input_packed)

output, _ = pad_packed_sequence(packed_output, batch_first=True)

Agregar BLEU score

Agregar optimización de GRU packed_seq

Agregar datos mezclados float16 en lugar de float32
